# Step 4: Create XML Files and Raw Metadata

Processes each HTML decision file: extracts structured content (Leitsätze, Rubrum, Tenor,
Gründe, Sondervota) and metadata (date, senate, ECLI, etc.), annotates paragraphs with
Tatbestand/Entscheidungsgründe (TB/EG) labels, and writes one XML file per decision plus
a raw metadata CSV.

The HTML source comes in two formats (`c-decision` for newer pages and `entscheidung` for
legacy pages), handled by separate parsing branches using functions from `helper.py`.

**Input:** `data/4_html_raw_232425/` – 857 HTML decision files  
**Output:** `data/5_xml/*.xml` – one XML file per decision  
**Output:** `data/6_metadata/metadata_232425_raw.csv` – raw metadata for all 857 decisions (39 columns)

The raw metadata is further cleaned and finalised in `5_export_metadata.ipynb`.

In [1]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from transformers import pipeline
from helper import *

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [2]:
# Define absolute paths for input/output directories and create them if they don't exist yet
PATH_MAIN = "/Users/kilian.lueders/code_lab_ur/corpus-update/"
PATH_INPUT  = PATH_MAIN + "data/4_html_raw_232425/"
PATH_OUTPUT_FILES = PATH_MAIN + "data/5_xml/"
PATH_OUTPUT_METADATA = PATH_MAIN + "data/6_metadata/"

if not os.path.exists(PATH_OUTPUT_FILES):
    os.mkdir(PATH_OUTPUT_FILES)

if not os.path.exists(PATH_OUTPUT_METADATA):
    os.mkdir(PATH_OUTPUT_METADATA)
    
files = os.listdir(PATH_INPUT)
print(len(files))

857


In [3]:
def create_entscheidung_xml(file):
    """
    Process a single decision HTML file: extract content and metadata, annotate
    Gründe paragraphs with TB/EG labels, save as XML, and return the metadata dict.

    The BVerfG website uses two HTML formats:
    - 'c-decision'  (div.c-decision)   – newer format
    - 'entscheidung' (div.entscheidung) – legacy format
    Both are handled by dedicated helper functions.

    Returns a metadata dict with 39 fields including dateiname, aktenzeichen,
    inAS, entscheidungsart, namenRichter, anzahlRichter, proceeding-type booleans,
    and all fields from get_metadata().
    """
    with open(PATH_INPUT + file, "r", encoding="utf-8") as fp:
        raw_file = fp.read()
    soup = BeautifulSoup(raw_file, "html.parser")

    # Extract sidebar metadata widgets (ECLI, citation, date, senate, etc.)
    content_div = soup.find("div", {"id": "content"})
    widgets = content_div.find_all("div", {"class": "l-widget"})
    metadata_raw = raw_metadata_widgets(widgets)
    metadata = get_metadata(metadata_raw)
    metadata['dateiname'] = file.replace(".html", "")

    # Detect which HTML format is present and parse accordingly
    decision_div = soup.find("div", {"class": "c-decision"})
    entscheidung_div = soup.find("div", {"class": "entscheidung"})

    if bool(decision_div):
        # Newer HTML format
        metadata['html_tag'] = "decision"
        judges_name, judges_count = get_judges_decision(decision_div)
        metadata['namenRichter'] = judges_name
        metadata['anzahlRichter'] = judges_count
        content_output, sondervota_output = get_content_decision(decision_div)
        metadata['abwmeinungen'] = int(len(sondervota_output) > 0)
        metadata.update(get_meta_content(content_output))
    elif bool(entscheidung_div):
        # Legacy HTML format
        metadata['html_tag'] = "entscheidung"
        judges_name, judges_count = get_judges_entscheidung(entscheidung_div)
        metadata['namenRichter'] = judges_name
        metadata['anzahlRichter'] = judges_count
        content_output, sondervota_output = get_content_entscheidung(entscheidung_div)
        metadata['abwmeinungen'] = int(len(sondervota_output) > 0)
        metadata.update(get_meta_content(content_output))

    # Flag for manual post-hoc corrections
    metadata['manuellErgaenzt'] = "0"
    # Extract Aktenzeichen from the parsed Rubrum text
    metadata['aktenzeichen'] = get_aktenzeichen(content_output)

    # Annotate each Gründe paragraph as Tatbestand (tb) or Entscheidungsgründe (eg)
    content_output = get_tbeg(content_output)
    # Sondervota are included in the XML output
    save_xml(content_output, sondervota_output, PATH_OUTPUT_FILES, file)

    return metadata


metadata_list = list()

# Process all files and collect metadata; print progress counter in place
for i, file in enumerate(files):
    print(f"{i} / {len(files)}       {file}", end="\r")
    metadata = create_entscheidung_xml(file)
    metadata_list.append(metadata)

# Add boolean columns for each BVerfG proceeding type (BvR, BvL, etc.)
metadata_df = pd.DataFrame(metadata_list)
metadata_df = add_proceeding_bools(metadata_df)

# Select and reorder to the final 39-column schema
metadata_df = metadata_df[['dateiname', 'aktenzeichen', 'inAS', 'fundstelle', 'band', 'ersteSeite',
       'letzteSeite', 'jahr', 'monat', 'tag', 'entscheidungsart',
       'spruchkoerper', 'gruende', 'abwmeinungen', 'bvr', 'bvl', 'bvq', 'bvc',
       'bve', 'bvf', 'bvo', 'bva', 'bvb', 'bvd', 'bvg', 'bvh', 'bvj', 'bvk',
       'bvm', 'bvn', 'bvp', 'bvt', 'pbvs', 'pbvu', 'pbvv', 'vz',
       'namenRichter', 'anzahlRichter', 'manuellErgaenzt']]

# Save raw metadata CSV (857 rows x 39 columns) – further cleaned in step 5
metadata_df.to_csv(PATH_OUTPUT_METADATA + "metadata_232425_raw.csv")
print(PATH_OUTPUT_METADATA + "metadata_232425_raw.csv")
print(metadata_df.shape)

print(f"Files in 5_xml/: {len(os.listdir(PATH_OUTPUT_FILES))}")

/Users/kilian.lueders/code_lab_ur/corpus-update/data/6_metadata/metadata_232425_raw.csv
(857, 39)
Files in 5_xml/: 857
